# Re-runs 2026-08 — flip-law Δµ, semantic steering, small-c seeding

Три прогона из утверждённого плана. Все — residual-stream, **транскодеры не нужны** (`circuit-tracer` не ставим).

| задача | что | сколько |
|---|---|---|
| **A** | 131 (Δµ-свип) → continuity-гейт → 132 (pool+train), оба концепта | ~1–2 ч |
| **B** | semantic steering, 8 концептов, L22/24/35 + аудит меток | ~2–4 ч |
| **C** | дозасев малых c (122 tier2) + пере-сборка 132 | ~20 мин |

Почему пере-прогон: июльский прогон шёл на `Qwen/Qwen3-4B-Base`, а дампы, из которых берутся направления и σ, — `Qwen/Qwen3-4B`. Подробности в `docs/RUN_TRACKER.md`.

## Перед запуском

1. **Запушить код в GitHub** (ноутбук клонит `main`; без пуша Colab возьмёт старые скрипты — PREFLIGHT это поймает).
2. Локально собрать бандлы:
   ```bash
   bash colab/make_data_bundle.sh        # ~/colab_data_bundle.tar.gz      (задачи A и C)
   bash colab/make_particles_bundle.sh   # ~/colab_particles_bundle.tar.gz (задача B)
   ```
3. Оба архива положить в корень `MyDrive`.
4. Colab: **Runtime → Change runtime type → GPU** (A100 лучше всего).
5. Ячейки сверху вниз: 0–3 это setup и preflight, дальше задачи. Порядок: **A → C** (C пере-собирает закон уже с Δµ), B независима и может идти в любой момент.

Гейт вшит внутрь `run_task2_fliplaw.sh`: если 131 не воспроизводит 122 по usage в пределах ±0.03, скрипт падает и 132 на плохих ячейках не соберётся.

In [ ]:
#@title 0. Конфиг + GPU
REPO_URL = "https://github.com/vitjuli/Mechanistic-Interpretability-of-Open-Source-LLMs.git"
BRANCH   = "main"
PROJ     = "/content/project"
BUNDLE       = "/content/drive/MyDrive/colab_data_bundle.tar.gz"       # задачи A, C
BUNDLE_PAIRS = "/content/drive/MyDrive/colab_particles_bundle.tar.gz"  # задача B
DRIVE_OUT    = "/content/drive/MyDrive/rerun_2026_08_out"              # куда складывать выходы

import os
os.system("nvidia-smi -L")

In [ ]:
#@title 1. Код: клон или pull + зависимости
import os
if os.path.exists(PROJ + "/.git"):
    !git -C $PROJ pull
else:
    !git clone --branch $BRANCH $REPO_URL $PROJ
%cd $PROJ
!pip -q install -r requirements.txt
!git -C $PROJ log -1 --oneline
import torch; print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())

In [ ]:
#@title 2. Данные: Drive + распаковка бандлов
from google.colab import drive; drive.mount('/content/drive')
%cd $PROJ
!tar -xzf "$BUNDLE" -C $PROJ
# бандл частиц нужен только для задачи B — если его нет, просто пропустится
!test -f "$BUNDLE_PAIRS" && tar -xzf "$BUNDLE_PAIRS" -C $PROJ || echo 'particles bundle не найден (задача B будет недоступна)'
!mkdir -p "$DRIVE_OUT"

In [ ]:
#@title 3. PREFLIGHT — есть ли фиксы в клоне и та ли модель у дампов
import re, glob, pathlib, numpy as np

problems = []
def chk(label, cond, hint=""):
    print(("OK   " if cond else "FAIL ") + label + ("" if cond else "   <- " + hint))
    if not cond: problems.append(label)

def read(p): return pathlib.Path(PROJ + "/" + p).read_text()

print("--- код ---")
s131 = read("scripts/131_delta_sweep_tier2.py")
m = re.search(r'--model_name",\s*default="([^"]+)"', s131)
chk(f"131 default model = {m.group(1) if m else '?'}", bool(m) and m.group(1) == "Qwen/Qwen3-4B",
    "фикс не запушен — запушь и перезапусти ячейку 1")
chk("131 сверяет модель с meta.npz", "model mismatch" in s131)
chk("есть continuity-гейт", pathlib.Path(PROJ + "/scripts/check_continuity_131_122.py").exists())
s122 = read("scripts/122_b1_steering_sweep.py")
chk("122 свипует delta", "delta_axis(H[trm]" in s122)
chk("122 умеет --c_anchor", "--c_anchor" in s122)
sdc = read("scripts/steering_decode_check.py")
md = re.search(r'--model",\s*default="([^"]+)"', sdc)
chk(f"decode-check default model = {md.group(1) if md else '?'}", bool(md) and md.group(1) == "Qwen/Qwen3-4B")
chk("decode-check берёт контраст из корпуса", "pair_classes" in sdc)

print("\n--- данные (задачи A, C) ---")
for cpt in ["B1_alpha_beta", "B1_grammar_number"]:
    d = pathlib.Path(PROJ) / "data/analysis/runD_v2" / cpt / "field_dump"
    has_meta = (d / "meta.npz").exists()
    chk(f"{cpt}: field_dump", has_meta, "распакуй colab_data_bundle.tar.gz")
    if has_meta:
        z = np.load(d / "meta.npz", allow_pickle=True)
        mn = str(z["model_name"]) if "model_name" in z.files else "(нет)"
        chk(f"{cpt}: модель дампа = {mn}", mn == "Qwen/Qwen3-4B")
        chk(f"{cpt}: res+grad 36 слоёв", len(glob.glob(str(d / 'res_L*.npy'))) == 36
            and len(glob.glob(str(d / 'grad_L*.npy'))) == 36)
    chk(f"{cpt}: cells_tier2.csv", (d.parent / "cells_tier2.csv").exists())
    chk(f"{cpt}: корпус", (pathlib.Path(PROJ) / f"data/prompts/{cpt}.jsonl").exists())

print("\n--- данные (задача B) ---")
pairs = sorted(glob.glob(PROJ + "/data/analysis/runD_v2/particles4_binary/*/field_dump/meta.npz"))
corp  = sorted(glob.glob(PROJ + "/data/prompts/particle_pairs/particles_*.jsonl"))
chk(f"дампы пар: {len(pairs)}/6", len(pairs) == 6, "распакуй colab_particles_bundle.tar.gz")
chk(f"корпуса пар: {len(corp)}/6", len(corp) == 6)

print()
if problems:
    print("НЕ ЗАПУСКАЙ задачи, пока это не закрыто:")
    for p in problems: print("  -", p)
else:
    print("всё на месте — можно запускать задачи")

## Задача A — flip-law: 131 → гейт → 132

Один скрипт делает всё для обоих концептов: Δµ-свип на правильной модели, continuity-проверка usage против 122, затем сборка закона для `pool` и `train`. 133 (whitening) выключен — он model-free и уже посчитан.

**Если ячейка упадёт на гейте — это штатное поведение.** Значит числа снова не сходятся, и читать `flip_law_*` нельзя. Смотри вывод гейта: он печатает худшие ячейки.

In [ ]:
%cd $PROJ
!bash colab/run_task2_fliplaw.sh

In [ ]:
#@title A-контроль: вердикт гейта + ключевые числа
import json, pathlib
ROOT = pathlib.Path(PROJ) / "data/analysis/runD_v2"

for cpt in ["B1_alpha_beta", "B1_grammar_number"]:
    print("=" * 70); print(cpt)
    !python -u scripts/check_continuity_131_122.py \
        --cells_122 {ROOT}/{cpt}/cells_tier2.csv \
        --cells_131 {ROOT}/{cpt}/cells_tier2_delta.csv \
        --concept {cpt} --out_json {ROOT}/{cpt}/continuity_gate.json
    for fs in ["pool", "train"]:
        f = ROOT / cpt / f"flip_law_{fs}" / "numbers_for_thesis.json"
        if not f.exists():
            print(f"  {fs}: нет файла"); continue
        d = json.load(open(f))
        rd, pl = d["realized_disp"], d["predicted_disp_linear"]
        ho = d.get("heldout_F", {})
        print(f"  {fs}: realized_disp MAE={rd.get('mae'):.4f} bias={rd.get('bias'):+.4f} n={rd.get('n')}"
              f" | predicted_disp_linear MAE={pl.get('mae'):.4f} (n={pl.get('n')})"
              f" | heldout degradation={ho.get('degradation', float('nan')):.4f}")

print("\nОжидания (kill-критерии): realized_disp.mae <= 0.08, |bias| < 0.02,")
print("heldout degradation < 0.05, predicted_disp_linear ≈ 0.028 (α/β) и 0.030 (grammar).")

## Задача B — semantic steering (8 концептов) + аудит меток

Гоняем **все 8**, а не только 6 пар: подмена чекпоинта задела и α/β с grammar. Контраст теперь берётся из самого корпуса, поэтому пары наконец меряются против партнёра, а не против случайного дистрактора.

Длинная ячейка (~2–4 ч). Выходы копируются на Drive внутри скрипта, так что разрыв сессии не смертелен.

In [ ]:
%cd $PROJ
!bash colab/run_semantic_steering.sh

In [ ]:
#@title B-контроль: аудит меток на СВЕЖЕМ csv (должно стать 128/128)
!python -u scripts/audit_semantic_steering_labels.py \
    --csv data/analysis/runD_v2/semantic_steering/semantic_steering_all.csv \
    --out data/analysis/runD_v2/semantic_steering/label_audit.csv

## Задача C — дозасев малых c + пере-сборка закона

Запускать **после A**. Меряем только новые значения `c ∈ {1/16, 1/8, 1/4}` на тех же слоях и тем же пулом, в отдельную папку `smallc/` (чтобы не затереть исходный `cells_tier2.csv`), затем пере-собираем 132 уже с тремя cells-файлами.

Это закрывает `<<demo>>` в §5.2 статьи: ячейки, где переход целиком лежит внутри линейного радиуса.

In [ ]:
import csv
R = "data/analysis/runD_v2"
%cd $PROJ

for CPT in ["B1_alpha_beta", "B1_grammar_number"]:
    layers = sorted({int(r["layer"]) for r in csv.DictReader(open(f"{R}/{CPT}/cells_tier2.csv"))})
    LSTR = " ".join(str(l) for l in layers)
    print(f"=== {CPT}: слои {LSTR}, c = 0.0625 0.125 0.25 ===")
    !python -u scripts/122_b1_steering_sweep.py \
        --corpus data/prompts/{CPT}.jsonl \
        --dump_dir {R}/{CPT}/field_dump \
        --tier 2 --tier2_layers {LSTR} \
        --t2_c_grid 0.0625 0.125 0.25 --c_anchor 1 \
        --dump_cells --out_dir {R}/{CPT}/smallc
    for FS in ["pool", "train"]:
        !python -u scripts/132_flip_law_assembly.py \
            --dump_dir {R}/{CPT}/field_dump \
            --cells {R}/{CPT}/cells_tier2.csv {R}/{CPT}/cells_tier2_delta.csv {R}/{CPT}/smallc/cells_tier2.csv \
            --concept {CPT} --F_split {FS} \
            --out_dir {R}/{CPT}/flip_law_{FS} --heldout_reps 50 \
            --split_seed 0 --train_frac 0.6 --shrink 0.1

## Забрать выходы

Кладём и на Drive (переживает разрыв сессии), и скачиваем в браузер.

In [ ]:
%cd $PROJ
!tar -czf /content/rerun_2026_08_out.tar.gz \
  data/analysis/runD_v2/*/flip_law_pool data/analysis/runD_v2/*/flip_law_train \
  data/analysis/runD_v2/*/cells_tier2_delta.csv \
  data/analysis/runD_v2/*/continuity_gate.json \
  data/analysis/runD_v2/*/smallc \
  data/analysis/runD_v2/semantic_steering 2>/dev/null || true
!cp /content/rerun_2026_08_out.tar.gz "$DRIVE_OUT/" && echo "→ $DRIVE_OUT"
!ls -lh /content/rerun_2026_08_out.tar.gz
from google.colab import files; files.download('/content/rerun_2026_08_out.tar.gz')

## Локально после скачивания

```bash
cd ~/Desktop/courses/thesis/project
tar -xzf ~/Downloads/rerun_2026_08_out.tar.gz

# гейт ещё раз, уже локально — он дешёвый и это последняя проверка перед текстом
python3 scripts/check_continuity_131_122.py \
  --cells_122 data/analysis/runD_v2/B1_alpha_beta/cells_tier2.csv \
  --cells_131 data/analysis/runD_v2/B1_alpha_beta/cells_tier2_delta.csv \
  --concept B1_alpha_beta
```

Только после зелёного гейта переносим числа в места, помеченные `<<131-rerun>>` в `~/Downloads/paper1_sec*.md`.